# Figure 5g — Per-Module Synergy: Combination vs. Single Agents (9-Module Breakdown)

## Overview
Computes baseline-corrected (vs. IL-1β) mean expression for 6 GO modules and 3 cellular-state modules per condition, then visualizes the stacked additive prediction alongside the observed combination effect (mean ± SEM) with Welch's t-test synergy significance markers.

## Inputs
- `../../Rdata/pub/mac_cmb0.1.sct.h5ad` — AnnData, combination treatments at 0.1 μM
- `../../Rdata/pub/mac_sin0.1.sct.h5ad` — AnnData, single-drug treatments at 0.1 μM
  - Both require `adata.obs['drug_name']`; `'inflammatory'` is the disease baseline

## Outputs
- `../output/main/Figure5g_module_synergy.pdf`

## Python environment
Tested with Python 3.9. Required packages: `scanpy`, `numpy`, `matplotlib`, `scipy`, `pandas`.
Run the final cell to record exact package versions.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.stats
import pandas as pd

matplotlib.rc("font", size=18)

# Paths — relative to this notebook's location (03_downstream_analysis/figures/)
DATA_DIR = Path("../../Rdata/pub")
OUT_DIR  = Path("../output/main")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
mac_cmb01 = sc.read_h5ad(DATA_DIR / "mac_cmb0.1.sct.h5ad")
mac_sin01  = sc.read_h5ad(DATA_DIR / "mac_sin0.1.sct.h5ad")

In [ ]:
# Gene module definitions — must match R/config.R GO_MODULES and STATE_MODULES
# 6 GO biological pathway modules
MODULE_GO = {
    "GO_Score1_ChondroDev":  ["Col2a1", "Acan", "Matn1", "Matn3", "Col27a1"],
    "GO_Score2_CartCondens": ["Sox9", "Ucma"],
    "GO_Score3_TissueHomeo": ["Pth1r", "Gm26633"],
    "GO_Score4_NOSynthase":  ["Fosl2"],
    "GO_Score5_ECMDisasm":   ["Mmp3", "Mmp13", "Adamts5"],
    "GO_Score6_InflamResp":  ["Il6", "Il17b", "Ccl2", "Cxcl5", "Cxcl1",
                               "Tlr2", "Tnfrsf1b"],
}

# 3 cellular-state safety modules
MODULE_STATE = {
    "Safe_Viability": ["Hprt", "Actb", "Gapdh", "B2m", "Ubc", "Ppia", "Rpl23"],
    "Safe_Stress":    ["Gadd45g", "Igfbp3"],
    "Safe_Prolif":    ["Ccnd3"],
}

MODULE_DICT = {**MODULE_GO, **MODULE_STATE}

# Baseline condition label (IL-1β treated)
COND_DISEASE = "inflammatory"

# Pre-compute mean module scores into .obs for fast access
for adata in [mac_cmb01, mac_sin01]:
    for mod_name, genes in MODULE_DICT.items():
        valid = [g for g in genes if g in adata.var_names]
        if valid:
            X = adata[:, valid].X
            if hasattr(X, "toarray"):
                X = X.toarray()
            adata.obs[mod_name] = np.mean(X, axis=1)
        else:
            adata.obs[mod_name] = 0.0

target_modules = list(MODULE_DICT.keys())

In [ ]:
def get_SEM(adata, pert, features, control_label=COND_DISEASE):
    ctrl  = adata.obs.loc[adata.obs["drug_name"] == control_label, features].values
    treat = adata.obs.loc[adata.obs["drug_name"] == pert, features].values
    ctrl_mean   = ctrl.mean(axis=0)
    corrected   = treat - ctrl_mean
    SEMs = {}
    for i, feat in enumerate(features):
        m  = np.mean(corrected[:, i])
        se = scipy.stats.sem(corrected[:, i])
        SEMs[feat] = (m, m - se, m + se)
    return SEMs

In [ ]:
from scipy.stats import ttest_ind

def get_corrected_arrays(adata, pert, features, control_label=COND_DISEASE):
    ctrl  = adata.obs.loc[adata.obs["drug_name"] == control_label, features].values
    treat = adata.obs.loc[adata.obs["drug_name"] == pert, features].values
    ctrl_mean = ctrl.mean(axis=0)
    corrected = treat - ctrl_mean
    return {feat: corrected[:, i] for i, feat in enumerate(features)}

def calc_synergy_pvalue(arr_A, arr_B, arr_combo):
    np.random.seed(42)
    n_cells    = len(arr_combo)
    sample_A   = np.random.choice(arr_A, size=n_cells, replace=True)
    sample_B   = np.random.choice(arr_B, size=n_cells, replace=True)
    arr_expected = sample_A + sample_B
    _, p_val   = ttest_ind(arr_combo, arr_expected, equal_var=False)
    # Only count synergy in the direction the combo shifted further from 0
    if abs(np.mean(arr_combo)) > abs(np.mean(arr_expected)):
        return p_val
    return 1.0

In [6]:
    
def add_bar_group_err(plot_df, x, combo, gene, ax, combo_color='tomato', seen=2, 
                      combo_CIs=0, single_CIs_1=0, single_CIs_2=0):
    
    x_g,y_g = combo.split('&')
    
    if seen == 2:
        n1 = plot_df.loc[x_g, gene]
        n2 = plot_df.loc[y_g, gene]
    elif seen == 1:
        n1 = plot_df.loc[x_g,gene]
        n2 = plot_df.loc['Naive', gene]

    true = plot_df.loc[combo, gene]

    ax.bar(x+1, n1, edgecolor='black', 
            linewidth=1.5, hatch="\\\\", color='lightgray')

    if n1>0:
        if n2>0:
            n_ = n1
        if n2<0:
            n_ = 0
    else:
        if n2>0:
            n_ = 0
        if n2<0:
            n_ = n1       
    ax.bar(x+1, n2, bottom=n_, edgecolor='black', 
            linewidth=1.5, color='lightgray', hatch='//')
        
    ax.bar(x, true, edgecolor='black', 
            linewidth=1, color=combo_color)
    plt.errorbar(x=x, y=true, yerr=combo_CIs, 
                 color='black')
    if n1>0:
        if n2>0:
            y_ = n1+n2
        if n2<0:
            y_ = n1
    else:
        if n2<0:
            y_ = n1+n2
        if n2>0:
            y_ = n2
        
    plt.errorbar(x=x+1, y=y_, yerr=np.abs(single_CIs_1)+np.abs(single_CIs_2), 
             color='black')
    

In [14]:
combo = "SB431542&SANT-1"
x_g = 'SB431542'
y_g = 'SANT-1'

combo_CIs = get_SEM(adata = mac_cmb01, pert = combo, features = target_modules)
single_CIs_1 = get_SEM(adata = mac_sin01, pert = x_g, features = target_modules)
single_CIs_2 = get_SEM(adata = mac_sin01, pert = y_g, features = target_modules)

# Fetch the raw single-cell arrays for statistical testing
arrays_combo = get_corrected_arrays(mac_cmb01, combo, target_modules)
arrays_single_1 = get_corrected_arrays(mac_sin01, x_g, target_modules)
arrays_single_2 = get_corrected_arrays(mac_sin01, y_g, target_modules)

# Create the DataFrame for plotting
mean_df = pd.DataFrame({
    'SB431542&SANT-1': [combo_CIs[feat][0] for feat in target_modules],
    'SB431542': [single_CIs_1[feat][0] for feat in target_modules],
    'SANT-1': [single_CIs_2[feat][0] for feat in target_modules]
}, index=target_modules)

plot_df = mean_df.T

In [ ]:
fig, ax = plt.subplots(figsize=(22, 6))
step = 3
xticks      = []
xticklabels = []
g1 = x_g
g2 = y_g

for itr, gene in enumerate(plot_df.columns):
    add_bar_group_err(
        plot_df, itr * step, combo, gene, ax, combo_color="firebrick",
        combo_CIs   = combo_CIs[gene][1]   - combo_CIs[gene][0],
        single_CIs_1 = single_CIs_1[gene][1] - single_CIs_1[gene][0],
        single_CIs_2 = single_CIs_2[gene][1] - single_CIs_2[gene][0],
    )

    p_val = calc_synergy_pvalue(
        arrays_single_1[gene], arrays_single_2[gene], arrays_combo[gene]
    )
    if p_val < 0.05:
        true_val  = plot_df.loc[combo, gene]
        combo_err = combo_CIs[gene][1] - combo_CIs[gene][0]
        y_pos = true_val + combo_err + 0.02 if true_val > 0 else true_val - combo_err - 0.05
        ax.text(itr * step, y_pos, "*", ha="center", va="center",
                fontsize=22, fontweight="bold")

    xticks.append(itr * step + 1)
    xticklabels.append(gene)

plt.legend(
    [f"Additive ({g1})", f"Additive ({g2})", "Combination"],
    ncol=3, loc="lower left"
)
plt.ylabel("Δ Module Expression (vs. Disease baseline)")
plt.xlabel("Module")
plt.title(f"Non-additive effect on module expression: {combo}")
plt.xticks(xticks, xticklabels, rotation=45, ha="right")
ax.axhline(y=0, color="k")

out_pdf = OUT_DIR / "Figure5g_module_synergy.pdf"
plt.savefig(out_pdf, format="pdf", bbox_inches="tight")
print("Saved:", out_pdf)

In [ ]:
# Environment reproducibility — run once after executing the notebook
import subprocess, sys, datetime

result   = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                          capture_output=True, text=True)
env_path = OUT_DIR.parent.parent / "session_info" / "figure5g_pip_freeze.txt"
env_path.parent.mkdir(parents=True, exist_ok=True)
with open(env_path, "w") as f:
    f.write(f"# figure5g.ipynb — pip freeze\n")
    f.write(f"# Recorded: {datetime.datetime.now().isoformat()}\n\n")
    f.write(result.stdout)
print(f"Environment recorded to: {env_path}")